In [1]:
import polars as pl
df = pl.scan_parquet('./dataset/test.parquet/test.parquet')
test_labels = pl.read_csv('./dataset/test.parquet/test_labels.csv')
unique_customers = df.select("customer_ID").unique().collect()


In [2]:
feat_ids = df.select("customer_ID").unique().collect().get_column("customer_ID")
lab_ids = test_labels.get_column("customer_ID").unique()

missing_labels = feat_ids.filter(~feat_ids.is_in(lab_ids))
missing_features = lab_ids.filter(~lab_ids.is_in(feat_ids))

print(f"IDs in features without labels: {len(missing_labels)}")
print(f"IDs in labels without features: {len(missing_features)}")

if len(missing_labels) == 0 and len(missing_features) == 0:
    print("The customer sets are identical.")

C:\Users\HP\AppData\Local\Temp\ipykernel_11740\138039564.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  missing_labels = feat_ids.filter(~feat_ids.is_in(lab_ids))
C:\Users\HP\AppData\Local\Temp\ipykernel_11740\138039564.py:5: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  missing_features = lab_ids.filter(~lab_ids.is_in(feat_ids))


IDs in features without labels: 0
IDs in labels without features: 0
The customer sets are identical.


In [3]:
# print(df.collect_schema())
print(test_labels.columns)

['customer_ID', 'prediction']


In [4]:
merged_data = (
    df.lazy().join(test_labels.lazy() if isinstance(test_labels , pl.DataFrame) else test_labels , on = "customer_ID" , how = "left").rename({"prediction" : "target"}).collect(streaming = True)
)

C:\Users\HP\AppData\Local\Temp\ipykernel_11740\2743195965.py:2: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df.lazy().join(test_labels.lazy() if isinstance(test_labels , pl.DataFrame) else test_labels , on = "customer_ID" , how = "left").rename({"prediction" : "target"}).collect(streaming = True)


In [5]:
print(merged_data.collect_schema())

Schema({'customer_ID': String, 'S_2': String, 'P_2': Float32, 'D_39': Int16, 'B_1': Float32, 'B_2': Float32, 'R_1': Float32, 'S_3': Float32, 'D_41': Float32, 'B_3': Float32, 'D_42': Float32, 'D_43': Float32, 'D_44': Int8, 'B_4': Int16, 'D_45': Float32, 'B_5': Float32, 'R_2': Int8, 'D_46': Float32, 'D_47': Float32, 'D_48': Float32, 'D_49': Int16, 'B_6': Float32, 'B_7': Float32, 'B_8': Float32, 'D_50': Float32, 'D_51': Int8, 'B_9': Float32, 'R_3': Int8, 'D_52': Float32, 'P_3': Float32, 'B_10': Float32, 'D_53': Float32, 'S_5': Float32, 'B_11': Float32, 'S_6': Int8, 'D_54': Float32, 'R_4': Int8, 'S_7': Float32, 'B_12': Float32, 'S_8': Int16, 'D_55': Float32, 'D_56': Float32, 'B_13': Float32, 'R_5': Int8, 'D_58': Float32, 'S_9': Float32, 'B_14': Float32, 'D_59': Int16, 'D_60': Float32, 'D_61': Float32, 'B_15': Float32, 'S_11': Int8, 'D_62': Float32, 'D_63': Int8, 'D_64': Int8, 'D_65': Int16, 'B_16': Int8, 'B_17': Float32, 'B_18': Float32, 'B_19': Int8, 'D_66': Int8, 'B_20': Int8, 'D_68': In

In [6]:
for col, dtype in merged_data.schema.items():
    if dtype == pl.String or col == "target": 
        print(f"Column: {col} | Type: {dtype}")

Column: customer_ID | Type: String
Column: S_2 | Type: String
Column: target | Type: Int64


In [7]:
numeric_cols = merged_data.select(pl.col(pl.Float32, pl.Float64, pl.Int32, pl.Int16, pl.Int8)).columns
numeric_cols = [c for c in numeric_cols if c not in ['customer_ID', 'target']]

cleaned_data = merged_data.with_columns([
    pl.col(numeric_cols).fill_null(-999)
])


cleaned_data = cleaned_data.with_columns([
    pl.col("S_2").fill_null(pl.date(1900, 1, 1))
])

In [8]:

all_features = [c for c in cleaned_data.columns if c not in ["customer_ID","target","S_2"]]
agg_exprs = [
    *[pl.col(c).mean().alias(f"{c}_mean") for c in all_features],
    *[pl.col(c).last().alias(f"{c}_last") for c in all_features],
    *[(pl.col(c).mean() - pl.col(c).last()).alias(f"{c}_delta") for c in all_features],
]


features_profiles = (
    cleaned_data.lazy()
    .group_by(["customer_ID","target"])
    .agg(agg_exprs)
    .collect(streaming = True)
)

features_profiles.write_parquet("final_test_data.parquet")

C:\Users\HP\AppData\Local\Temp\ipykernel_11740\3192826517.py:13: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming = True)


In [10]:
test_data = pl.scan_parquet('./dataset/test.parquet/final_test_data.parquet')
print(test_data.collect())

shape: (924_621, 566)
┌────────────┬────────┬──────────┬───────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ customer_I ┆ target ┆ P_2_mean ┆ D_39_mean ┆ … ┆ D_142_delt ┆ D_143_delt ┆ D_144_del ┆ D_145_del │
│ D          ┆ ---    ┆ ---      ┆ ---       ┆   ┆ a          ┆ a          ┆ ta        ┆ ta        │
│ ---        ┆ i64    ┆ f32      ┆ f64       ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
│ str        ┆        ┆          ┆           ┆   ┆ f32        ┆ f64        ┆ f32       ┆ f64       │
╞════════════╪════════╪══════════╪═══════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 13d3d7735c ┆ 0      ┆ 0.896681 ┆ 0.0       ┆ … ┆ 0.0        ┆ 0.0        ┆ -0.001782 ┆ 0.0       │
│ a99fc7f5a7 ┆        ┆          ┆           ┆   ┆            ┆            ┆           ┆           │
│ 81239c4494 ┆        ┆          ┆           ┆   ┆            ┆            ┆           ┆           │
│ …          ┆        ┆          ┆           ┆   ┆            ┆      